In [0]:
# Step 1: Set up the tables used by the pipeline
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

bronze_table = f"{catalog}.{schema_name}.streaming_orders_bronze"
silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"
audit_table = f"{catalog}.{schema_name}.streaming_data_quality_audit"

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("Pipeline run:", run_id)
print("Bronze:", bronze_table)
print("Silver:", silver_table)
print("Audit:", audit_table)

In [0]:
# Step 2: Make sure the source table exists before processing
if not spark.catalog.tableExists(bronze_table):
    raise ValueError(
        f"Bronze table not found: {bronze_table}. "
        "Run Notebook 09 first."
    )

bronze_df = spark.table(bronze_table)

print("Bronze records:", bronze_df.count())
display(bronze_df)

In [0]:
# Step 3: Clean the Bronze data before merging into Silver
prepared_df = (
    bronze_df
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
    .dropDuplicates(["order_id"])
)

valid_condition = (
    F.col("order_id").isNotNull()
    & F.col("order_date").isNotNull()
    & F.col("product_id").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("unit_price").isNotNull()
    & (F.col("unit_price") >= 0)
)

valid_orders = prepared_df.filter(valid_condition)
invalid_orders = prepared_df.filter(~valid_condition)

print("Valid records:", valid_orders.count())
print("Invalid records:", invalid_orders.count())

In [0]:
# Step 4: Create Silver if needed, then merge new records
if not spark.catalog.tableExists(silver_table):
    (
        valid_orders.limit(0)
        .write
        .format("delta")
        .saveAsTable(silver_table)
    )

if valid_orders.limit(1).count() > 0:
    silver_delta = DeltaTable.forName(spark, silver_table)

    (
        silver_delta.alias("target")
        .merge(
            valid_orders.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Silver processing complete.")
print("Silver records:", spark.table(silver_table).count())

In [0]:
# Step 5: Check the final Silver table
silver_df = spark.table(silver_table)

quality_checks = [
    (
        "order_id_not_null",
        silver_df.filter(F.col("order_id").isNull()).count() == 0
    ),
    (
        "order_id_unique",
        silver_df.groupBy("order_id").count()
        .filter(F.col("count") > 1).count() == 0
    ),
    (
        "quantity_positive",
        silver_df.filter(
            F.col("quantity").isNull() | (F.col("quantity") <= 0)
        ).count() == 0
    ),
    (
        "order_date_valid",
        silver_df.filter(F.col("order_date").isNull()).count() == 0
    )
]

for name, passed in quality_checks:
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

In [0]:
# Step 6: Summarize this run for easier monitoring
failed_checks = sum(not passed for _, passed in quality_checks)

summary = {
    "run_id": run_id,
    "bronze_rows": bronze_df.count(),
    "valid_source_rows": valid_orders.count(),
    "invalid_source_rows": invalid_orders.count(),
    "silver_rows": silver_df.count(),
    "quality_checks": len(quality_checks),
    "failed_checks": failed_checks,
    "status": "SUCCESS" if failed_checks == 0 else "FAILED"
}

print(summary)

In [0]:
# Step 7: Keep a history of pipeline runs
summary_df = spark.createDataFrame([summary])

run_log_table = f"{catalog}.{schema_name}.pipeline_run_log"

if not spark.catalog.tableExists(run_log_table):
    summary_df.write.format("delta").saveAsTable(run_log_table)
else:
    summary_df.write.format("delta").mode("append").saveAsTable(run_log_table)

display(
    spark.table(run_log_table)
    .orderBy(F.col("run_id").desc())
)